# Code-Mixed Summarizer — Colab Training

Trains IndicBART in 3 phases on a Colab GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

**Time (rough):** Phase 1 ~3–6h, Phase 2 ~2–4h, Phase 3 ~2–4h on T4 with full data (varies).

**After training:** Copy `artifacts/checkpoints/phase3/` to your PC for Streamlit/CLI inference.

In [ ]:
# Quick test vs full training
QUICK_TEST = False  # True = ~200 samples/lang, 1 epoch — sanity check only

# Save checkpoints to Google Drive so they survive session disconnect
USE_DRIVE = True
DRIVE_FOLDER = "codemix_summarizer"  # created under My Drive/

# Colab T4 (15GB): batch 16 often OOM — use smaller batch
COLAB_BATCH_SIZE = 4
COLAB_GRAD_ACCUM = 4

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CKPT_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER}/checkpoints"
    print("Checkpoints will sync to:", DRIVE_CKPT_ROOT)
else:
    DRIVE_CKPT_ROOT = None

## 1. Clone project & install dependencies

In [ ]:
import os
REPO = "/content/Multilingual-code-mixed-text-Summarizer"
if not os.path.isdir(REPO):
    !git clone https://github.com/raunaqmittal/Multilingual-code-mixed-text-Summarizer.git {REPO}
%cd {REPO}
!pip install -q -r requirements.txt

## 2. Prepare data

Pick **one** option:
- **A (recommended):** Run the data pipeline on Colab (~15–20 min, downloads from Hugging Face).
- **B:** You already prepared data on your PC — zip `artifacts/data/preprocessed` + `synthetic_codemixed` + `dictionaries`, upload to Drive, unzip into `artifacts/data/`.

In [ ]:
# Option A: build data on Colab (skip if preprocessed/ already exists)
RUN_DATA_PIPELINE = True

preprocessed_ok = os.path.isfile("artifacts/data/preprocessed/Hindi/train_clean.csv")
synthetic_ok = os.path.isfile("artifacts/data/synthetic_codemixed/Hindi/train_synthetic.csv")

if preprocessed_ok and synthetic_ok:
    print("Data already present — skipping download pipeline.")
elif RUN_DATA_PIPELINE:
    !python -m src.components.data.dataset_loader
    !python -m src.components.data.preprocessor
    !python -m src.components.data.synthetic_generator
else:
    raise FileNotFoundError(
        "No preprocessed data found. Set RUN_DATA_PIPELINE=True or upload artifacts/data from your PC."
    )

In [ ]:
# Patch training YAMLs for Colab VRAM + optional quick test
import pathlib
import yaml

def patch_phase_config(path, epochs_override=None):
    with open(path) as f:
        cfg = yaml.safe_load(f)
    cfg["batch_size"] = COLAB_BATCH_SIZE
    cfg["gradient_accumulation_steps"] = COLAB_GRAD_ACCUM
    cfg["mixed_precision"] = True
    if epochs_override is not None:
        cfg["epochs"] = epochs_override
    with open(path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print("Patched", path, "->", cfg)

epochs = 1 if QUICK_TEST else None
patch_phase_config("src/configs/phase1_train.yaml", epochs)
patch_phase_config("src/configs/phase2_train.yaml", epochs)
patch_phase_config("src/configs/phase3_train.yaml", epochs)

if QUICK_TEST:
    p1 = pathlib.Path("src/pipelines/train_phase1.py").read_text()
    p1 = p1.replace("MAX_SAMPLES_PER_LANGUAGE = None", "MAX_SAMPLES_PER_LANGUAGE = 200")
    pathlib.Path("src/pipelines/train_phase1.py").write_text(p1)
    for name in ["train_phase2.py", "train_phase3.py"]:
        t = pathlib.Path(f"src/pipelines/{name}").read_text()
        t = t.replace("MAX_SAMPLES_PER_SYNTHETIC = None", "MAX_SAMPLES_PER_SYNTHETIC = 200")
        t = t.replace("MAX_SAMPLES_REAL = None", "MAX_SAMPLES_REAL = 100")
        if "phase3" in name:
            t = t.replace("MAX_SAMPLES_PER_ILSUM = None", "MAX_SAMPLES_PER_ILSUM = 100")
        pathlib.Path(f"src/pipelines/{name}").write_text(t)
    print("QUICK_TEST sample limits applied.")

## 3. Phase 1 — Monolingual (ILSUM)

Run this cell, wait until it finishes, then continue to Phase 2.

In [ ]:
!python -m src.pipelines.train_phase1

In [ ]:
import shutil
if USE_DRIVE and os.path.isdir("artifacts/checkpoints/phase1"):
    dst = f"{DRIVE_CKPT_ROOT}/phase1"
    os.makedirs(dst, exist_ok=True)
    shutil.copytree("artifacts/checkpoints/phase1", dst, dirs_exist_ok=True)
    print("Backed up phase1 to Drive:", dst)

## 4. Phase 2 — Code-mixed adaptation

Requires `artifacts/checkpoints/phase1/config.json`.

In [ ]:
# If session restarted, restore phase1 from Drive:
if USE_DRIVE and not os.path.isfile("artifacts/checkpoints/phase1/config.json"):
    src = f"{DRIVE_CKPT_ROOT}/phase1"
    if os.path.isfile(os.path.join(src, "config.json")):
        os.makedirs("artifacts/checkpoints", exist_ok=True)
        shutil.copytree(src, "artifacts/checkpoints/phase1", dirs_exist_ok=True)
        print("Restored phase1 from Drive")

!python -m src.pipelines.train_phase2

In [ ]:
if USE_DRIVE and os.path.isdir("artifacts/checkpoints/phase2"):
    dst = f"{DRIVE_CKPT_ROOT}/phase2"
    os.makedirs(dst, exist_ok=True)
    shutil.copytree("artifacts/checkpoints/phase2", dst, dirs_exist_ok=True)
    print("Backed up phase2 to Drive:", dst)

## 5. Phase 3 — Stabilization

Requires `artifacts/checkpoints/phase2/config.json`.

In [ ]:
if USE_DRIVE and not os.path.isfile("artifacts/checkpoints/phase2/config.json"):
    src = f"{DRIVE_CKPT_ROOT}/phase2"
    if os.path.isfile(os.path.join(src, "config.json")):
        shutil.copytree(src, "artifacts/checkpoints/phase2", dirs_exist_ok=True)
        print("Restored phase2 from Drive")

!python -m src.pipelines.train_phase3

In [ ]:
if USE_DRIVE and os.path.isdir("artifacts/checkpoints/phase3"):
    dst = f"{DRIVE_CKPT_ROOT}/phase3"
    os.makedirs(dst, exist_ok=True)
    shutil.copytree("artifacts/checkpoints/phase3", dst, dirs_exist_ok=True)
    print("Backed up phase3 to Drive:", dst)

## 6. Download checkpoint to your PC

On your Windows machine, copy the folder:

`Google Drive/codemix_summarizer/checkpoints/phase3/`

→

`Multilingual-code-mixed-text-Summarizer/artifacts/checkpoints/phase3/`

Then run locally:
```bash
streamlit run app.py
```

In [ ]:
# Zip phase3 for easy download from Colab sidebar
!apt-get -qq install zip > /dev/null
!cd artifacts/checkpoints && zip -r /content/phase3_checkpoint.zip phase3
from google.colab import files
files.download("/content/phase3_checkpoint.zip")

In [ ]:
# Quick inference test on Colab
!python -m src.pipelines.inference_pipeline --text "aaj bharat ne cricket match jeeta aur poori team ne bahut achha khela"